# Week 1: Hindi tokenizer sampling on Kaggle

This notebook clones the public project, verifies the Kaggle/Hugging Face environment, samples Hindi text, trains the SentencePiece tokenizer, compares tokenizer fertility, and optionally uploads tokenizer artifacts. Keep the Accelerator set to **None** for every cell; reserve the T4 for the later `train` command.

## Settings checklist
- [ ] Accelerator: **None**
- [ ] Internet: **On**
- [ ] Kaggle Secret named **HF_TOKEN** added (token is never printed)

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path('/kaggle/working/HINDILLMf')
if not repo_dir.is_dir():
    subprocess.run([
        'git', 'clone',
        'https://github.com/nayaksuvamsuryakanta-a11y/HINDILLMf.git',
        str(repo_dir),
    ], check=True)
%cd /kaggle/working/HINDILLMf
!ls -la

In [ ]:
!python scripts/check_env.py

# Only install missing pure-Python support packages. Never install or upgrade
# torch, transformers, datasets, or numpy in this notebook.
import importlib.util
import subprocess
import sys

report = subprocess.run(
    [sys.executable, 'scripts/check_env.py'],
    check=True, capture_output=True, text=True,
)
missing = {line.split(':', 1)[0].strip() for line in report.stdout.splitlines()
           if line.rstrip().endswith(': missing')}
pure_python = {'huggingface_hub': 'huggingface_hub'}
packages = [pure_python[module] for module in sorted(missing)
            if module in pure_python and importlib.util.find_spec(module) is None]
# check_env.py does not list dotenv, but check_remotes.py needs python-dotenv.
if importlib.util.find_spec('dotenv') is None:
    packages.append('python-dotenv')
for package in dict.fromkeys(packages):
    print(f'Installing missing pure-Python support package: {package}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', package], check=True)
if not packages:
    print('No missing pure-Python support packages to install.')
print('Protected packages (torch, transformers, datasets, numpy) were not installed or upgraded.')

In [ ]:
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
assert len(hf_token) > 10, 'HF_TOKEN secret is missing or invalid.'
os.environ['HF_TOKEN'] = hf_token
Path('.env').write_text(
    f'HF_TOKEN={hf_token}\n'
    'HF_REPO_ID=Nsuvam/hindi-llm-170m-base\n'
    'WORK_DIR=work\n',
    encoding='utf-8',
)
print('HF_TOKEN loaded from Kaggle Secrets and .env written; token value hidden.')

In [ ]:
!python scripts/check_remotes.py

In [ ]:
!df -h /kaggle/working
!if [ -d work ]; then du -sh work; else echo 'work directory does not exist yet'; fi

## Sampling time and restart behavior

Sampling three million lines may take several hours, and Kaggle sessions end at 12 hours. The tokenizer sampling stage does **not** resume from a manifest or saved cursor: each invocation starts streaming from the beginning, accumulates its sample in memory, overwrites `work/tokenizer_sample.txt` after sampling finishes, and then trains SentencePiece from that newly written file. If a session ends during sampling, rerunning the tokenizer command starts sampling again from the beginning; it does not continue the previous sample. The T4 remains unused because this notebook's Accelerator is set to None.

In [ ]:
# The repository defaults HF sources to disabled. Enable only CulturaX here;
# ensure its dataset terms/access have been approved for this Kaggle account.
import json
from pathlib import Path

config_path = Path('configs/sources.json')
config = json.loads(config_path.read_text(encoding='utf-8'))
for source in config['sources']:
    if source.get('dataset'):
        source['enabled'] = (source['name'] == 'culturax_hi')
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

!python hindi_llm.py tokenizer --source hf --online --vocab 32000 --max_sample_lines 3000000 --work work

In [ ]:
import sentencepiece as spm
from transformers import AutoTokenizer

sample_path = 'work/tokenizer_sample.txt'
with open(sample_path, encoding='utf-8') as handle:
    lines = [line.strip() for line in handle if line.strip()][:2000]
if not lines:
    raise RuntimeError(f'No non-empty sample lines found in {sample_path}')

sp_tokenizer = spm.SentencePieceProcessor(model_file='work/hindi_bpe.model')
tinyllama_tokenizer = AutoTokenizer.from_pretrained('TinyLlama/TinyLlama-1.1B-Chat-v1.0')
word_count = sum(len(line.split()) for line in lines)
sp_piece_count = sum(len(sp_tokenizer.encode(line, out_type=int)) for line in lines)
tinyllama_token_count = sum(len(tinyllama_tokenizer.encode(line, add_special_tokens=False)) for line in lines)
print(f'Lines compared: {len(lines)}')
print(f'Words: {word_count}')
print(f'SentencePiece tokens per word: {sp_piece_count / word_count:.4f}')
print(f'TinyLlama tokens per word: {tinyllama_token_count / word_count:.4f}')

In [ ]:
# Optional: uploads tokenizer artifacts to the configured private model repository.
from huggingface_hub import HfApi
import os
from pathlib import Path

repo_id = 'Nsuvam/hindi-llm-170m-base'
artifact_paths = [
    'work/hindi_bpe.model',
    'work/hindi_bpe.vocab',
    'work/tokenizer_meta.json',
]
try:
    api = HfApi(token=os.environ['HF_TOKEN'])
    for path in artifact_paths:
        api.upload_file(
            path_or_fileobj=path,
            path_in_repo=Path(path).name,
            repo_id=repo_id,
            repo_type='model',
        )
    print(f'Uploaded tokenizer artifacts to {repo_id}.')
except Exception as exc:
    print(f'Optional upload failed ({type(exc).__name__}): {exc}')
    print('Check that the repository exists, HF_TOKEN has write access, and Internet is enabled; rerun this cell after fixing access.')

## Next steps

Run the `process` stage in a separate **CPU-only session** with Accelerator **None**, saving and checking the manifest and binary files between sessions. Reserve the T4 for the `train` command in a later T4 session. See the [Kaggle setup guide](../docs/KAGGLE_SETUP.md) and [execution plan](../PLAN_EXECUTION.md) for commands and milestones.